# B5 · Attention, explained with image captioning: how an image becomes a sentence
**Series B: Deep Learning for Molecules, Post 5 of 7** &nbsp;|&nbsp; companion *primer* notebook

The full tutorial of this post (*Image Captioning with CLIP Prefixes (ClipCap)*, Prof. V. S. Batista, CHEM 584, adapted from the inference notebook of the ClipCap authors) combines two large pre-trained models, **CLIP** (an image encoder) and **GPT-2** (a language model), through a small learned **bridge**: the image becomes a *prefix* of ten vectors that GPT-2 reads as the beginning of a sentence and continues. It needs a GPU runtime, a 0.6 GB weight file and web images. This primer rebuilds **every mechanism in miniature, with NumPy only and in seconds**: a contrastive CLIP, the mapping network, a causal-attention language model with hand-written backpropagation, the three decoders, and the probing experiments of the tutorial. Part II then runs the primer's *own decoders* on the real ClipCap model.

> **Two parts.** **Part I** needs only NumPy, pandas and matplotlib and runs in about 15 seconds on a CPU. **Part II** needs PyTorch, Transformers and the ClipCap weights (a GPU runtime is recommended).
>
> **Reading the code.** Every section starts with a short **Stop and predict** question: answer it before running the cell.

| Section | What you do |
|---|---|
| 1 | a toy world of scenes and captions, and CLIP in miniature (contrastive training) |
| 2 | the ClipCap model: mapping network, prefix, causal attention; shapes and parameter counts |
| 3 | train it, check the gradients, and reach the entropy floor of the grammar |
| 4 | three decoders: greedy, top-$p$, beam; the bug of the original `generate2`; why beam search can win |
| 5 | probing: does the image matter? interpolation, what the prefix looks like, where the attention goes |
| 6 | scoring captions with CLIP |
| **Part II** | the primer's decoders on the real ClipCap model |

## 0. Setup

In [ ]:
import math, time, importlib.util
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True, linewidth=120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")
def softmax(x, axis=-1):
    x = x - x.max(axis, keepdims=True); e = np.exp(x); return e / e.sum(axis, keepdims=True)

## 1. A toy world, and CLIP in miniature
**The world.** A *scene* is an object (dog, cat, car, bird) in a place (park, street, beach): 12 scenes. Its **image embedding** is a vector of $E=12$ numbers, the sum of an object vector and a place vector plus noise: a stand-in for the 512 numbers that CLIP computes. A **caption** has six tokens from a 12-word vocabulary, `[a|the] OBJECT [in|at] the PLACE .`, with the article chosen with probabilities $0.7/0.3$ and the preposition $0.6/0.4$. These two random choices are the only uncertainty in a caption once the image is known, so the best possible loss is their entropy per token, which we can compute.
**CLIP in miniature.** CLIP learns an image tower and a text tower so that the embedding of an image is **close to its caption and far from other captions** (the contrastive, InfoNCE, loss). Here both towers are linear maps; captions enter as bags of words.
**Stop and predict:** what is the lowest possible per-token cross-entropy of a model that knows the image?

In [ ]:
OBJ = ["dog", "cat", "car", "bird"]; PLACE = ["park", "street", "beach"]
VOCAB = ["a", "the", "dog", "cat", "car", "bird", "in", "at", "park", "street", "beach", "."]; V = len(VOCAB); TOK = {w: i for i, w in enumerate(VOCAB)}; STOP = TOK["."]
E, P, D, H, L = 12, 3, 16, 32, 6              # CLIP-like embedding size, prefix length, GPT-like width, mapping hidden size, caption length
_r = np.random.RandomState(0); obj_vec, place_vec = _r.randn(4, E), _r.randn(3, E)
def scene_embed(o, p, rng, noise=0.2): return obj_vec[o] + place_vec[p] + noise * rng.standard_normal(E)
def sample_caption(o, p, rng):
    art = rng.choice(["a", "the"], p=[0.7, 0.3]); prep = rng.choice(["in", "at"], p=[0.6, 0.4])
    return [TOK[w] for w in (art, OBJ[o], prep, "the", PLACE[p], ".")]
def make_batch(n, rng):
    o, p = rng.integers(0, 4, n), rng.integers(0, 3, n)
    return np.stack([scene_embed(a, b, rng) for a, b in zip(o, p)]), np.array([sample_caption(a, b, rng) for a, b in zip(o, p)]), o, p
H_art = -(0.7 * math.log(0.7) + 0.3 * math.log(0.3)); H_prep = -(0.6 * math.log(0.6) + 0.4 * math.log(0.4)); FLOOR = (H_art + H_prep) / L
print(f"entropy of the article {H_art:.4f} nats and of the preposition {H_prep:.4f} nats -> lowest possible loss per token: {FLOOR:.4f}")
c_demo, y_demo, o_demo, p_demo = make_batch(3, np.random.default_rng(0)); print("example captions:", [" ".join(VOCAB[t] for t in row) for row in y_demo])
assert abs(FLOOR - 0.2139) < 1e-3

In [ ]:
def l2n(x): n = np.linalg.norm(x, axis=1, keepdims=True); return x / n, n
def clip_train(steps=400, lr=0.05, tau=0.1, bs=48, seed=0):
    r = np.random.default_rng(seed); Wi = r.normal(0, 0.3, (E, E)); Wt = r.normal(0, 0.3, (V, E)); m = [np.zeros_like(Wi), np.zeros_like(Wt)]; v = [np.zeros_like(Wi), np.zeros_like(Wt)]; hist = []
    for t in range(1, steps + 1):
        c, y, o, p_ = make_batch(bs, r); bow = np.zeros((bs, V))
        for i in range(bs): np.add.at(bow[i], y[i], 1.0)
        sid = o * 3 + p_; ni, nin = l2n(c @ Wi); nt, ntn = l2n(bow @ Wt); S = ni @ nt.T / tau
        same = sid[:, None] == sid[None, :]; tgt = same / same.sum(1, keepdims=True)                 # captions of the same scene are not negatives of each other
        Pi, Pt = softmax(S, 1), softmax(S.T, 1); hist.append(-0.5 * ((tgt * np.log(Pi + 1e-12)).sum(1).mean() + (tgt * np.log(Pt + 1e-12)).sum(1).mean()))
        dS = 0.5 * (Pi - tgt) / bs + 0.5 * ((Pt - tgt) / bs).T; dni, dnt = dS @ nt / tau, dS.T @ ni / tau
        dui = (dni - (dni * ni).sum(1, keepdims=True) * ni) / nin; dut = (dnt - (dnt * nt).sum(1, keepdims=True) * nt) / ntn
        for k, (W, g) in enumerate(((Wi, c.T @ dui), (Wt, bow.T @ dut))):
            m[k] = 0.9 * m[k] + 0.1 * g; v[k] = 0.999 * v[k] + 0.001 * g ** 2; W -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
    return Wi, Wt, hist
Wi, Wt, clip_hist = clip_train(); print(f"contrastive loss {clip_hist[0]:.3f} -> {np.mean(clip_hist[-20:]):.3f}")
bow_of = lambda tokens: np.bincount(tokens, minlength=V).astype(float)
def clip_sim(c, tokens): a, b = c @ Wi, bow_of(tokens) @ Wt; return float(a @ b / np.linalg.norm(a) / np.linalg.norm(b))   # cosine similarity of the embeddings
canon = [[TOK[w] for w in ("a", OBJ[o], "in", "the", PLACE[q], ".")] for o in range(4) for q in range(3)]
rg = np.random.default_rng(4); M = np.array([[clip_sim(scene_embed(s // 3, s % 3, rg), cap) for cap in canon] for s in range(12)])
hit = int((M.argmax(1) == np.arange(12)).sum()); matched, mismatched = float(np.trace(M) / 12), float((M.sum() - np.trace(M)) / 132)
print(f"retrieval: the right caption is the most similar one for {hit} of 12 scenes | mean cosine: matched {matched:.3f}, mismatched {mismatched:.3f}")
assert hit == 12 and matched > mismatched + 0.5

## 2. The ClipCap model: mapping network, prefix and causal attention
**The bridge.** A mapping network (here a multilayer perceptron $E\to H\to P\cdot d$ with a $\tanh$) turns the image embedding $c$ into $P$ vectors of size $d$, the **prefix**. In the tutorial $E=512$, $H=3840$, $P=10$, $d=768$.
**The language model.** The input sequence is `[prefix, token embeddings]` plus position embeddings; one layer of **causal self-attention** lets each position look at itself and at the positions before it, $A=\mathrm{softmax}\big(QK^T/\sqrt d+\text{mask}\big)$, output $AV$, then a residual connection and a projection to the vocabulary. The prediction for the next token is read at the **last** position. In training, the input is the prefix and the caption without its last token, and the targets are the six caption tokens. The image enters only through the prefix.
**Stop and predict:** how many parameters does the real mapping network have, and what is the shape of the input of GPT-2 for a caption of $L$ tokens?

In [ ]:
def init(seed=0):
    r = np.random.default_rng(seed); n = lambda *s, sc=0.15: r.normal(0, sc, s)
    return {"W1": n(E, H, sc=0.3), "b1": np.zeros(H), "W2": n(H, P * D, sc=0.2), "b2": np.zeros(P * D), "tok": n(V, D, sc=0.5), "pos": n(P + 10, D, sc=0.3),
            "Wq": n(D, D), "Wk": n(D, D), "Wv": n(D, D), "Wo": n(D, D), "Wout": n(D, V, sc=0.3), "bout": np.zeros(V)}
def forward(p, c, tokens_in):
    """c: (B, E) image embeddings, tokens_in: (B, Lin) -> logits (B, P + Lin, V) for every position, and a cache"""
    B, Lin = tokens_in.shape; h = np.tanh(c @ p["W1"] + p["b1"]); prefix = (h @ p["W2"] + p["b2"]).reshape(B, P, D)
    X = np.concatenate([prefix, p["tok"][tokens_in]], 1) + p["pos"][:P + Lin]; T = P + Lin
    Q, K, Vv = X @ p["Wq"], X @ p["Wk"], X @ p["Wv"]; S = Q @ K.transpose(0, 2, 1) / np.sqrt(D); S = np.where(np.triu(np.ones((T, T), bool), 1), -1e9, S); A = softmax(S)
    O = A @ Vv; X1 = X + O @ p["Wo"]; return X1 @ p["Wout"] + p["bout"], dict(c=c, h=h, X=X, Q=Q, K=K, Vv=Vv, A=A, O=O, X1=X1, tokens_in=tokens_in)
def loss_grads(p, c, y):
    B = len(c); logits, ca = forward(p, c, y[:, :-1]); Pr = softmax(logits[:, P - 1:, :]); idx = (np.arange(B)[:, None], np.arange(L)[None, :], y)
    loss = -np.log(Pr[idx] + 1e-12).mean(); dl = Pr.copy(); dl[idx] -= 1; dl /= B * L; dlog = np.zeros_like(logits); dlog[:, P - 1:, :] = dl
    X1, X, A, O, Q, K, Vv = (ca[k] for k in ("X1", "X", "A", "O", "Q", "K", "Vv")); T = X.shape[1]; g = {}
    g["Wout"] = np.einsum("btd,btv->dv", X1, dlog); g["bout"] = dlog.sum((0, 1)); dX1 = dlog @ p["Wout"].T
    g["Wo"] = np.einsum("btd,bte->de", O, dX1); dO = dX1 @ p["Wo"].T; dA = dO @ Vv.transpose(0, 2, 1); dV = A.transpose(0, 2, 1) @ dO
    dS = A * (dA - (dA * A).sum(-1, keepdims=True)); dQ = dS @ K / np.sqrt(D); dK = dS.transpose(0, 2, 1) @ Q / np.sqrt(D)
    g["Wq"], g["Wk"], g["Wv"] = (np.einsum("btd,bte->de", X, d_) for d_ in (dQ, dK, dV))
    dX = dX1 + dQ @ p["Wq"].T + dK @ p["Wk"].T + dV @ p["Wv"].T; g["pos"] = np.zeros_like(p["pos"]); g["pos"][:T] = dX.sum(0)
    g["tok"] = np.zeros_like(p["tok"]); np.add.at(g["tok"], ca["tokens_in"], dX[:, P:, :])
    dpre = dX[:, :P, :].reshape(B, P * D); g["W2"] = ca["h"].T @ dpre; g["b2"] = dpre.sum(0); dh = (dpre @ p["W2"].T) * (1 - ca["h"] ** 2); g["W1"] = ca["c"].T @ dh; g["b1"] = dh.sum(0)
    return loss, g
p0 = init(0); n_par = sum(v.size for v in p0.values()); print("parameters of the miniature model:", n_par, "| shapes:", {k: v.shape for k, v in p0.items() if k in ("W1", "W2", "Wq", "Wout")})
# the real model, by arithmetic
E_r, H_r, P_r, d_r = 512, 3840, 10, 768
n_map = (E_r * H_r + H_r) + (H_r * P_r * d_r + P_r * d_r)
blk = (3 * d_r * d_r + 3 * d_r) + (d_r * d_r + d_r) + (d_r * 3072 + 3072) + (3072 * d_r + d_r) + 4 * d_r
n_gpt2 = 12 * blk + 50257 * d_r + 1024 * d_r + 2 * d_r
print(f"real mapping network 512 -> 3840 -> 7680: {n_map:,} parameters | GPT-2 small (12 blocks, tied output layer): {n_gpt2:,} parameters | the prefix has {P_r * d_r:,} numbers")
print(f"input of GPT-2 for a caption of L tokens: (10 + L) x 768; the attention matrix is (10 + L) x (10 + L), e.g. {(10 + 12) ** 2} entries for L = 12")
assert n_map == 31_468_800 and n_gpt2 == 124_439_808

## 3. Training, with a gradient check and an exact target
We train the whole model (mapping network and language model) with Adam on captions sampled from the grammar. Two checks tell us the implementation is right: the **gradients** by hand against finite differences, and the **final loss** against the entropy floor of Section 1: a model that has learned everything that can be learned reaches it, and none can go below it.
**Stop and predict:** will the loss fall below the floor? Will it reach it?

In [ ]:
rng = np.random.default_rng(1); c_b, y_b, _, _ = make_batch(8, rng); loss_b, g_b = loss_grads(p0, c_b, y_b); worst = 0.0; r2_ = np.random.default_rng(5)
for k in p0:
    for _ in range(6):
        idx = tuple(r2_.integers(0, s) for s in p0[k].shape); a_, b_ = ({n: v.copy() for n, v in p0.items()} for _ in range(2)); a_[k][idx] += 1e-6; b_[k][idx] -= 1e-6
        worst = max(worst, abs((loss_grads(a_, c_b, y_b)[0] - loss_grads(b_, c_b, y_b)[0]) / 2e-6 - g_b[k][idx]))
print(f"gradient check on {6 * len(p0)} random parameters (every array): largest |finite difference - backpropagation| = {worst:.1e}"); assert worst < 1e-7
def adam_train(p, steps=600, lr=0.01, bs=64, seed=2):
    m = {k: np.zeros_like(v) for k, v in p.items()}; v = {k: np.zeros_like(x) for k, x in p.items()}; rg_ = np.random.default_rng(seed); hist = []
    for t in range(1, steps + 1):
        l, g = loss_grads(p, *make_batch(bs, rg_)[:2]); hist.append(l)
        for k in p: m[k] = 0.9 * m[k] + 0.1 * g[k]; v[k] = 0.999 * v[k] + 0.001 * g[k] ** 2; p[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
    return hist
model = init(0); t0 = time.perf_counter(); hist = adam_train(model); final = float(np.mean(hist[-20:]))
print(f"trained in {time.perf_counter() - t0:.1f} s: loss {hist[0]:.3f} -> {final:.4f} | floor {FLOOR:.4f} | perplexity per token {math.exp(final):.3f}")
assert FLOOR - 0.01 < final < FLOOR + 0.03
fig, ax = plt.subplots(figsize=(6.4, 3.6)); ax.plot(hist, color=COLORS["blue"], label="training loss"); ax.axhline(FLOOR, color=COLORS["red"], ls="--", label=f"entropy floor {FLOOR:.3f}")
ax.set(xlabel="step", ylabel="cross-entropy per token", title="The miniature ClipCap reaches the floor"); ax.legend(); plt.tight_layout(); plt.show()

## 4. Decoding
The model gives a distribution over the next token. **Greedy** takes the most probable token; **top-$p$ (nucleus) sampling** keeps the smallest set of tokens whose probability adds up to $p$ (after dividing the logits by a temperature $T$), renormalizes and *draws*; **beam search** keeps the $B$ best partial sentences, ranked by their mean log-probability per token, and returns the best complete one. The decoders below take a function `step(tokens)` that returns the log-probabilities of the next token, so the same code serves the toy model here and the real model in Part II.
**Stop and predict:** the original tutorial code `generate2` is meant to be top-$p$ sampling: it removes the tokens outside the nucleus and then chooses the token with `argmax`. What does it compute?

In [ ]:
def greedy(step, stop=STOP, maxlen=8):
    toks = []
    for _ in range(maxlen):
        t = int(np.argmax(step(toks))); toks.append(t)
        if t == stop: break
    return toks
def nucleus_filter(logits, top_p):
    pr = softmax(logits); order = np.argsort(-pr); cum = np.cumsum(pr[order]); remove = cum > top_p; remove[1:] = remove[:-1].copy(); remove[0] = False     # keep the token that crosses the threshold
    out = logits.copy(); out[order[remove]] = -np.inf; return out
def top_p_sample(step, top_p, T, rng, stop=STOP, maxlen=8):
    toks = []
    for _ in range(maxlen):
        pr = softmax(nucleus_filter(step(toks) / max(T, 1e-6), top_p)); t = int(rng.choice(len(pr), p=pr)); toks.append(t)       # a random DRAW from the nucleus
        if t == stop: break
    return toks
def beam_search(step, B=5, stop=STOP, maxlen=8):
    """Returns [(tokens, mean log-probability per token)] for the B best sentences."""
    beams = [([], 0.0, False)]
    for _ in range(maxlen):
        cand = []
        for toks, sc, done in beams:
            if done: cand.append((toks, sc, True)); continue
            lp = step(toks)
            for t in np.argsort(-lp)[:B]: cand.append((toks + [int(t)], sc + float(lp[t]), int(t) == stop))
        cand.sort(key=lambda b: -b[1] / len(b[0])); beams = cand[:B]
        if all(b[2] for b in beams): break
    return [(b[0], b[1] / len(b[0])) for b in beams]
def make_step(p, cvec):
    def step(tokens):
        lg, _ = forward(p, cvec[None, :], np.array(tokens, int).reshape(1, -1)); return np.log(softmax(lg[0, -1]))
    return step
txt = lambda toks: " ".join(VOCAB[t] for t in toks)
rg = np.random.default_rng(9); rows = []; right = lambda s, o, q: (OBJ[o] in s.split()) and (PLACE[q] in s.split())
for o in range(4):
    for q in range(3):
        step = make_step(model, scene_embed(o, q, rg)); g_ = txt(greedy(step)); b_ = txt(beam_search(step, 5)[0][0]); b1 = txt(beam_search(step, 1)[0][0]); assert b1 == g_
        rows.append((f"{OBJ[o]} / {PLACE[q]}", g_, b_, right(g_, o, q), right(b_, o, q)))
tab = pd.DataFrame(rows, columns=["scene", "greedy", "beam search (B = 5)", "greedy right", "beam right"]); print(tab.to_string(index=False))
print(f"\nobject and place right: greedy {tab['greedy right'].sum()} of 12, beam {tab['beam right'].sum()} of 12; beam size 1 equals greedy in every scene")
assert tab["greedy right"].all() and tab["beam right"].all()
step = make_step(model, scene_embed(0, 0, np.random.default_rng(3))); print("\nthe five beams for a dog in the park, with their mean log-probability per token:"); [print(f"  {sc:7.3f}  {txt(t)}") for t, sc in beam_search(step, 5)]

In [ ]:
# the bug of generate2: a nucleus filter followed by argmax is greedy decoding, whatever p and T
rr = np.random.default_rng(0); same = 0; n_tr = 2000
for _ in range(n_tr):
    lg = rr.normal(0, rr.uniform(0.2, 4.0), V); same += int(np.argmax(nucleus_filter(lg / rr.uniform(0.1, 3.0), rr.uniform(0.05, 0.99))) == np.argmax(lg))
print(f"nucleus filter + argmax equals greedy in {same} of {n_tr} random trials (random logits, p in (0.05, 0.99), T in (0.1, 3))"); assert same == n_tr
# temperature: 40 sampled captions of a dog in the park, nucleus p = 0.9
step = make_step(model, scene_embed(0, 0, np.random.default_rng(3))); rows = []
for T in (0.5, 1.0, 2.0, 4.0):
    rr = np.random.default_rng(11); caps = [txt(top_p_sample(step, 0.9, T, rr)) for _ in range(40)]; rows.append((T, len(set(caps)), 100 * np.mean([right(c, 0, 0) for c in caps])))
tt = pd.DataFrame(rows, columns=["temperature", "distinct captions of 40", "object and place right (%)"]); print("\n" + tt.to_string(index=False))
assert tt.iloc[0, 2] > 95 and tt.iloc[-1, 2] < 80 and tt.iloc[-1, 1] > tt.iloc[0, 1]
# why beam search can beat greedy: a sentence of two words; first word a (0.5), b (0.4), c (0.1); after a the second word is spread over five options, after b it is certain
LM = {(): {"a": 0.5, "b": 0.4, "c": 0.1}, ("a",): {w: 0.2 for w in "xyzwv"}, ("b",): {"x": 1.0}, ("c",): {"x": 1.0}}
gr = ("a", max(LM[("a",)], key=LM[("a",)].get)); sp = lambda s: LM[()][s[0]] * LM[(s[0],)][s[1]]; allseq = [(a, b) for a in "abc" for b in LM[(a,)]]; bst = max(allseq, key=sp)
print(f"\ngreedy picks {gr} with probability {sp(gr):.2f}; the most probable sentence is {bst} with probability {sp(bst):.2f}; a beam of 2 keeps 'b' and finds it")
assert abs(sp(gr) - 0.1) < 1e-12 and abs(sp(bst) - 0.4) < 1e-12

## 5. Probing: does the image matter, and where does the attention go?
If the image enters only through the prefix, **replacing the prefix** should change the caption. We caption a dog in the park with (a) its own embedding, (b) the embedding of a car on the beach, (c) an all-zero embedding and (d) a random vector of the same length; then we **interpolate** between the two scenes, look at what the prefix vectors resemble, and measure how much of each caption position's attention goes to the prefix.
**Stop and predict:** what should (b), (c) and (d) produce?

In [ ]:
rg = np.random.default_rng(21); cA, cB = scene_embed(0, 0, rg), scene_embed(2, 2, rg); rand = rg.standard_normal(E); rand *= np.linalg.norm(cA) / np.linalg.norm(rand)
caps5 = {}
for lab, cc in (("(a) own embedding (dog, park)", cA), ("(b) embedding of another scene (car, beach)", cB), ("(c) all-zero embedding", np.zeros(E)), ("(d) random vector of the same length", rand)):
    caps5[lab[:3]] = txt(beam_search(make_step(model, cc), 5)[0][0]); print(f"{lab:46s}: {caps5[lab[:3]]}")
assert "dog" in caps5["(a)"] and "park" in caps5["(a)"] and "car" in caps5["(b)"] and "beach" in caps5["(b)"]
lam = np.linspace(0, 1, 9); interp = [txt(beam_search(make_step(model, (1 - l) * cA + l * cB), 5)[0][0]) for l in lam]
print("\ninterpolation from the dog in the park to the car on the beach:"); [print(f"  lambda = {l:.2f}: {s}") for l, s in zip(lam, interp)]
assert "dog" in interp[0] and "car" in interp[-1] and "park" in interp[0] and "beach" in interp[-1]
prefix = (np.tanh(cA[None] @ model["W1"] + model["b1"]) @ model["W2"] + model["b2"]).reshape(P, D); tn = np.linalg.norm(model["tok"], axis=1)
cs = (prefix / np.linalg.norm(prefix, axis=1, keepdims=True)) @ (model["tok"] / np.linalg.norm(model["tok"], axis=1, keepdims=True)).T
print(f"\naverage length of a token embedding {tn.mean():.2f} | lengths of the {P} prefix vectors {np.linalg.norm(prefix, axis=1).round(2)} | nearest token of each prefix vector: {[(VOCAB[int(cs[k].argmax())], round(float(cs[k].max()), 2)) for k in range(P)]}")
toks = [TOK[w] for w in ("a", "dog", "in", "the", "park")]; A = forward(model, cA[None], np.array([toks]))[1]["A"][0]; mass = A[P:, :P].sum(1)
print("attention mass that each caption position puts on the prefix (positions: a, dog, in, the, park):", mass.round(2), "| next token to predict: dog, in, the, park, '.'")
assert np.linalg.norm(prefix, axis=1).mean() > 1.5 * tn.mean()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8)); im = ax[0].imshow(A, cmap="Blues", vmin=0, vmax=1)
labels = [f"p{k + 1}" for k in range(P)] + ["a", "dog", "in", "the", "park"]; ax[0].set_xticks(range(len(labels))); ax[0].set_xticklabels(labels); ax[0].set_yticks(range(len(labels))); ax[0].set_yticklabels(labels)
ax[0].set(xlabel="attended position", ylabel="query position", title="(a) causal self-attention weights"); plt.colorbar(im, ax=ax[0], fraction=0.046)
ax[1].bar(range(5), mass, color=COLORS["blue"]); ax[1].set_xticks(range(5)); ax[1].set_xticklabels(["a", "dog", "in", "the", "park"]); ax[1].set(ylabel="attention mass on the prefix", title="(b) how much each position looks at the image", ylim=(0, 1.05))
plt.tight_layout(); plt.show()

## 6. Scoring captions with CLIP
Without reference captions, the similarity between the image embedding and the caption embedding, the **CLIP score**, tells how well a caption matches the image. Here we use the miniature CLIP of Section 1 to score the five beams of a dog in the park and to **re-rank** them.
**Stop and predict:** which of the five beams will the score reject, and which differences will it not see?

In [ ]:
cvec = scene_embed(0, 0, np.random.default_rng(3)); bm = beam_search(make_step(model, cvec), 5)
sc = pd.DataFrame([(txt(t), s, clip_sim(cvec, t)) for t, s in bm], columns=["beam", "mean log-probability", "CLIP-mini similarity"]); print(sc.round(3).to_string(index=False))
worst_beam = sc.loc[sc["CLIP-mini similarity"].idxmin(), "beam"]; print(f"\nthe lowest similarity belongs to: '{worst_beam}' (the only beam with the wrong place); articles and prepositions barely change the score")
assert "beach" in worst_beam and sc["CLIP-mini similarity"].min() < sc["CLIP-mini similarity"].max() - 0.3

## 7. Your turn
**Exercise 1.** Change the beam size to 1, 2, 3 and 10. How do the captions and their scores change, and what is the running time?
**Exercise 2.** Scale the embedding by 0.5 and by 2 before the mapping network (the real models were trained with *unnormalized* embeddings). What happens to the captions?
**Exercise 3.** Add a second attention head, or a second layer. Does the loss still reach the floor, and does the attention to the prefix change?
**Exercise 4.** Add noise of size 0.5 to the scene embeddings at test time. Where do the captions fail first, the object or the place?
**Exercise 5.** Make the prefix length $P=1$. Can the model still caption all 12 scenes?

## 8. Self-check

In [ ]:
assert abs(FLOOR - 0.2139) < 1e-3 and worst < 1e-7 and FLOOR - 0.01 < final < FLOOR + 0.03 and n_map == 31_468_800 and n_gpt2 == 124_439_808
assert tab["greedy right"].all() and same == n_tr and "car" in caps5["(b)"] and hit == 12 and "beach" in worst_beam
print("All checks passed: you reproduced the numbers in the post.")

# Part II. The primer's decoders on the real ClipCap model
> **Read this first.** Part II loads the real model: CLIP ViT-B/32 and GPT-2 from the Hugging Face hub, and the ClipCap weights (the *Conceptual Captions* checkpoint, 0.6 GB, from Google Drive with `gdown`), builds the mapping network and the prefix, and wraps GPT-2 in the same `step(tokens)` interface as Section 4: **the greedy and beam-search functions of Part I then caption real images unchanged**. The cells compare the results with the numbers of the tutorial's saved output (image 562207 of the COCO validation set). Use a GPU runtime (**Runtime → Change runtime type**); on a CPU the captions take longer. If the download from Google Drive fails (it limits popular files), the cell says how to fetch the file by hand. Each stage is guarded and the last cell lists the checks that passed.

In [ ]:
!pip install -q transformers gdown pillow

In [ ]:
HAVE_REAL = all(importlib.util.find_spec(m) is not None for m in ("torch", "transformers", "gdown", "PIL"))
checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
if HAVE_REAL:
    import os, urllib.request, torch, torch.nn as nn, gdown, PIL.Image
    from transformers import AutoTokenizer, GPT2LMHeadModel, CLIPModel, CLIPProcessor
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", device)
    WORKDIR = "/content" if "google.colab" in __import__("sys").modules else "."
    WEIGHTS_ID, WEIGHTS_PATH = "14pXWwB4Zm82rsDdvbGguLfx9F8aM7ovT", os.path.join(WORKDIR, "conceptual_captions_weights.pt")
    real_ok = False
    try:
        if not (os.path.isfile(WEIGHTS_PATH) and os.path.getsize(WEIGHTS_PATH) > 100e6): gdown.download(id=WEIGHTS_ID, output=WEIGHTS_PATH, quiet=False)
        real_ok = os.path.isfile(WEIGHTS_PATH) and os.path.getsize(WEIGHTS_PATH) > 100e6
        if not real_ok: print("The download failed: fetch https://drive.google.com/uc?id=" + WEIGHTS_ID + " by hand, put the file at", WEIGHTS_PATH, "and re-run this cell.")
    except Exception as e: print("The download failed:", type(e).__name__, e)
else: print("PyTorch, Transformers, gdown or Pillow is not available: run the install cell above and re-run this cell.")

In [ ]:
if HAVE_REAL and real_ok:
    try:
        class RealClipCap(nn.Module):
            def __init__(self):
                super().__init__(); self.gpt = GPT2LMHeadModel.from_pretrained("gpt2")
                self.clip_project = nn.Sequential(); self.clip_project.model = nn.Sequential(nn.Linear(512, 3840), nn.Tanh(), nn.Linear(3840, 7680))
            def prefix_of(self, clip_embedding): return self.clip_project.model(clip_embedding).view(-1, 10, 768)
        real = RealClipCap(); state = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=True)
        state = {k: v for k, v in state.items() if not k.endswith((".attn.bias", ".attn.masked_bias"))}                   # entries of older versions that current ones do not use
        missing, unexpected = real.load_state_dict(state, strict=False); missing = [k for k in missing if k != "gpt.lm_head.weight"]
        check("every parameter of the model is found in the weight file (nothing missing, nothing unexpected)", not missing and not unexpected)
        real = real.float().to(device).eval(); tok = AutoTokenizer.from_pretrained("gpt2")
        n_map_real = sum(q.numel() for q in real.clip_project.parameters()); n_gpt_real = sum(q.numel() for q in real.gpt.parameters())
        print(f"mapping network {n_map_real:,} parameters (arithmetic of Part I: {n_map:,}) | GPT-2 {n_gpt_real:,} (arithmetic: {n_gpt2:,}; the output layer is tied to the embedding)")
        check("the mapping network has the parameter count of Part I", n_map_real == n_map); check("GPT-2 has the parameter count of Part I", n_gpt_real == n_gpt2)
        clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").float().to(device).eval(); clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        STOP_REAL = tok.encode(".")[0]; check("the stop token '.' has id 13", STOP_REAL == 13)
    except Exception as e: real_ok = False; print("stage could not be completed:", type(e).__name__, e)

In [ ]:
if HAVE_REAL and real_ok:
    try:
        def fetch(image_id):
            name = f"COCO_val2014_000000{image_id}.jpg"; path = os.path.join(WORKDIR, name)
            if not os.path.isfile(path): urllib.request.urlretrieve(f"https://images.cocodataset.org/val2014/{name}", path)
            return PIL.Image.open(path).convert("RGB")
        @torch.no_grad()
        def clip_embedding(img):
            px = clip_proc(images=img, return_tensors="pt")["pixel_values"].to(device); return clip_model.visual_projection(clip_model.vision_model(pixel_values=px).pooler_output).float()
        def real_step_for(emb):
            with torch.no_grad(): prefix = real.prefix_of(emb)
            @torch.no_grad()
            def step(tokens):                                                              # log-probabilities of the next token, as in Section 4
                x = prefix if not tokens else torch.cat((prefix, real.gpt.transformer.wte(torch.tensor([tokens], device=device))), 1)
                return torch.log_softmax(real.gpt(inputs_embeds=x).logits[0, -1].float(), -1).cpu().numpy()
            return step
        emb_a, emb_b = clip_embedding(fetch("562207")), clip_embedding(fetch("579664"))
        check("the CLIP embedding has shape (1, 512) and the prefix (1, 10, 768)", tuple(emb_a.shape) == (1, 512) and tuple(real.prefix_of(emb_a).shape) == (1, 10, 768))
        step_a = real_step_for(emb_a); t0 = time.perf_counter()
        g_real = tok.decode(greedy(step_a, stop=STOP_REAL, maxlen=67)).strip(); beams_real = [(tok.decode(t).strip(), s) for t, s in beam_search(step_a, 5, stop=STOP_REAL, maxlen=67)]
        print(f"image 562207, the primer's decoders on the real model ({time.perf_counter() - t0:.1f} s):\n  greedy: {g_real}\n  beam search, the five beams:"); [print(f"   {s:7.3f}  {t}") for t, s in beams_real]
        print("the tutorial's saved output: greedy 'person and i with an elephant.'; beams -0.777 'person and i with an elephant.', -0.922 'tourists with an elephant in the river.', ...")
        check("greedy caption equals the tutorial's ('person and i with an elephant.')", g_real == "person and i with an elephant.")
        check("the best beam equals the tutorial's and its score is within 0.03 of -0.777", beams_real[0][0] == "person and i with an elephant." and abs(beams_real[0][1] + 0.777) < 0.03)
        check("the beam of size 1 equals greedy decoding on the real model", tok.decode(beam_search(step_a, 1, stop=STOP_REAL, maxlen=67)[0][0]).strip() == g_real)
        swapped = tok.decode(beam_search(real_step_for(emb_b), 5, stop=STOP_REAL, maxlen=67)[0][0]).strip()
        print("the prefix of the other image (579664) gives:", swapped, "| the tutorial's saved output: 'bananas on a market stall.'")
        check("a prefix from another image gives that image's caption (the tutorial's 'bananas on a market stall.')", swapped == "bananas on a market stall.")
        pre = real.prefix_of(emb_a)[0].detach(); E_tok = real.gpt.transformer.wte.weight.detach()
        ratio = float(pre.norm(dim=-1).mean() / E_tok.norm(dim=-1).mean()); print(f"prefix vectors: mean length {float(pre.norm(dim=-1).mean()):.1f} against {float(E_tok.norm(dim=-1).mean()):.2f} for a token embedding (ratio {ratio:.1f}; the tutorial: 13 to 20 against 3.97)")
        check("the prefix vectors are more than twice as long as a typical token embedding (as in the miniature of Section 5)", ratio > 2)
    except Exception as e: print("stage could not be completed:", type(e).__name__, e)

In [ ]:
if HAVE_REAL:
    ok = [lab for lab, passed in checks if passed]; bad = [lab for lab, passed in checks if not passed]
    print(f"Part II: {len(ok)} of {len(checks)} checks passed")
    for lab in bad: print("  not passed:", lab)

### What to look for
* **Counts.** The mapping network has $31{,}468{,}800$ parameters and GPT-2 small $124{,}439{,}808$: the arithmetic of Part I.
* **The same decoders.** Greedy decoding, beam search and the beam of size 1 of Part I run unchanged on the real model and give the tutorial's captions; a different score of the best beam within the tolerance is rounding, a different caption would mean a difference in the decoders or in the model.
* **The prefix controls the caption.** The prefix of another image gives that image's caption, and the prefix vectors are several times longer than real token embeddings, exactly as in the miniature.